<a href="https://colab.research.google.com/github/zyf-hitsz/PytorchLearning/blob/main/%E7%A5%9E%E7%BB%8F%E7%BD%91%E7%BB%9C/%E6%B1%A0%E5%8C%96%E5%B1%82/PoolingLayers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#池化
---
池化是每个通道单独进行的，主要有一下一些作用：

1：用于降低特征图尺寸以减小计算量。

2：扩大有效感受野：即经过池化（可以粗糙地理解为下采样）后，下一层一个位置对应原始输入中更大的区域，因此越深层的神经元可以逐渐感知：
$$边缘→纹理→局部结构→物体部件→完整物体$$

3：获得一定的平移鲁棒性：如最大池化提取局部最大值，那么该局部的非最大值发生小幅度平移不会改变池化结果，以这样的原理池化可以提供一定的平移鲁棒性。

Pool1d、Pool2d、Pool3d和不同维度的卷积是类似的，只是在滑动的维度上不同，适用于不同维度的输入数据，没有太多本质的区别。




---
##最大池化MaxPoolNd



In [ ]:
#以二维最大池化为例
class torch.nn.MaxPool2d(kernel_size, stride=None, padding=0, dilation=1, return_indices=False, ceil_mode=False)
#kernel_size：卷积核尺寸，池化窗口的大小，可以是单个整数（如2代表2×2），也可以写成（3，4）等。
#stride:步长，窗口滑动的距离，默认值等于kernel_size，stride越小，输出尺寸越大。
#padding:在输入边界四周填充数的层数，增加padding可以减缓特征图尺寸的缩减，更多地保留边缘特征。
#dilation:控制读取输入数据时窗口内元素之间的距离，默认为1，若>1，则是"跳跃式读取"
#return_indices:返回索引，如果为True，除了返回最大值，还会返回最大值在原始输入中的索引位置。
#ceil_mode (向上取整模式)：默认为 False（向下取整 floor），边缘不满一整个池化窗的数据会被直接舍弃；
    #如果设为 True，计算输出形状时会使用ceil。这能确保即使最后的滑动窗口不足 kernel_size 大小，其中的输入像素也会被包含在输出中。

### **Dilation的一点解释**
当 `dilation > 1` 时，池化窗口覆盖的范围会变大，导致输出尺寸减小得更快。

In [3]:
import torch
import torch.nn as nn

# 输入是一个 5x5 的矩阵
input_tensor = torch.randn(1, 1, 5, 5)

# 1. 普通 3x3 池化 (dilation=1)
pool_normal = nn.MaxPool2d(kernel_size=3, stride=1, dilation=1)
output_normal = pool_normal(input_tensor)
# 输出尺寸计算: 5 - (3-1)*1 - 1 = 3 -> 3x3
print(f"dilation=1 输出: {output_normal}")

# 2. 空洞池化 (dilation=2)
# 此时 kernel_size 还是 3，但元素间有间距，实际覆盖范围变成了 3+(3-1)*(2-1) = 5
pool_dilation = nn.MaxPool2d(kernel_size=3, stride=1, dilation=2)
output_dilation = pool_dilation(input_tensor)
# 输出尺寸计算: 5 - (3-1)*2 - 1 = 0 -> 1x1
print(f"dilation=2 输出: {output_dilation}")

dilation=1 输出: tensor([[[[2.1571, 2.1571, 0.8091],
          [0.9730, 0.9730, 0.8091],
          [0.9004, 0.7866, 0.7866]]]])
dilation=2 输出: tensor([[[[0.7139]]]])


#### 实验对比：比较Dilation 和 扩大 Kernel Size？
我们将对比 `kernel=2, dilation=2` (跨度大) 和 `kernel=2, dilation=1` (跨度小) 的结果。

In [7]:
import torch
import torch.nn as nn

# 定义一个 4x4 输入
input_data = torch.tensor([[[[1.0, 2.0, 3.0, 4.0],
                          [5.0, 6.0, 7.0, 8.0],
                            [9.0, 10.0, 11.0, 12.0],
                            [13.0, 14.0, 15.0, 16.0]]]])

# 情况 A: kernel=2, dilation=1 (紧凑采样)
pool_a = nn.MaxPool2d(kernel_size=2, stride=1, dilation=1)
# 情况 B: kernel=2, dilation=2 (稀疏采样，覆盖 3x3 区域)
pool_b = nn.MaxPool2d(kernel_size=2, stride=1, dilation=2)
pool_c = nn.MaxPool2d(kernel_size=3, stride=1, dilation=1)


print(f"输入矩阵尺寸: {input_data.shape[2:]}")
print(f"情况 A (d=1) 输出尺寸: {pool_a(input_data).shape[2:]} (局部采样)")
print(f"情况 B (d=2) 输出尺寸: {pool_b(input_data).shape[2:]} (跨域采样)")
print(f"情况 C (d=1) 输出尺寸: {pool_c(input_data).shape[2:]} (跨域采样)")

# 观察数值差异
print("\n情况 A 输出:", pool_a(input_data))
print("情况 B 输出:", pool_b(input_data))
print("情况 c 输出:", pool_c(input_data))

输入矩阵尺寸: torch.Size([4, 4])
情况 A (d=1) 输出尺寸: torch.Size([3, 3]) (局部采样)
情况 B (d=2) 输出尺寸: torch.Size([2, 2]) (跨域采样)
情况 C (d=1) 输出尺寸: torch.Size([2, 2]) (跨域采样)

情况 A 输出: tensor([[[[ 6.,  7.,  8.],
          [10., 11., 12.],
          [14., 15., 16.]]]])
情况 B 输出: tensor([[[[11., 12.],
          [15., 16.]]]])
情况 c 输出: tensor([[[[11., 12.],
          [15., 16.]]]])


在上面的例子中，我们发现空洞池化和扩大核的效果可能在某些时候是一样的，下面这个例子可以直观看到二者的区别。
#### 核心区别演示：空洞 vs 实心
我们将构建一个特殊的输入：在 $3 \times 3$ 区域的正中心放一个极大值。实心窗口（k=3）能看到它，但空洞窗口（k=2, d=2）会跳过它。

形象的解释：池化核比作手掌，扩大size是手掌变大了，覆盖到更多的数据，中间不会有遗漏；空洞卷积则更像张开的手指，覆盖范围更大了，但是手指缝之间会遗漏一些数据。

In [11]:
import torch
import torch.nn as nn

# 创建一个 3x3 的输入，只有中心点是 100，其余是 1
# [[1, 1, 1],
#  [1, 100, 1],
#  [1, 1, 1]]
test_input = torch.ones((1, 1, 3, 3))
test_input[0, 0, 1, 1] = 100.0

# 情况 1：实心窗口，覆盖 3x3 区域内所有点
pool_solid = nn.MaxPool2d(kernel_size=3, stride=1, dilation=1)
# 情况 2：空洞窗口，覆盖 3x3 区域，但只看四个角点 (0,0), (0,2), (2,0), (2,2)
pool_dilated = nn.MaxPool2d(kernel_size=2, stride=1, dilation=2)

print("输入矩阵:")
print(test_input[0,0])
print(f"\n实心窗口 (k=3, d=1) 输出: {pool_solid(test_input).item()}")
print(f"空洞窗口 (k=2, d=2) 输出: {pool_dilated(test_input).item()}")

输入矩阵:
tensor([[  1.,   1.,   1.],
        [  1., 100.,   1.],
        [  1.,   1.,   1.]])

实心窗口 (k=3, d=1) 输出: 100.0  <-- 捕捉到了中心的100
空洞窗口 (k=2, d=2) 输出: 1.0  <-- 跳过了中心，只看到周边的1


### 展示 return_indices 的作用
当 `return_indices=True` 时，模型会返回池化结果以及最大值的索引。

In [12]:
import torch
import torch.nn as nn

# 创建一个简单的 4x4 输入
input_data = torch.tensor([[[[1.0, 2.0, 3.0, 4.0],
                            [5.0, 6.0, 7.0, 8.0],
                            [9.0, 10.0, 11.0, 12.0],
                            [13.0, 14.0, 15.0, 16.0]]]])

# 设置 return_indices=True
pool = nn.MaxPool2d(kernel_size=2, stride=2, return_indices=True)
output, indices = pool(input_data)

print("池化后的输出:")
print(output)
print("\n对应的索引 (indices):")
print(indices)

池化后的输出:
tensor([[[[ 6.,  8.],
          [14., 16.]]]])

对应的索引 (indices):
tensor([[[[ 5,  7],
          [13, 15]]]])


---
##平均池化AvgPoolNd
平均池化和最大池化几乎一样，只是计算窗口内的算术平均值而不是只选取最大值。

In [ ]:
import torch.nn as nn
# 二维平均池化定义
# 注意：它没有 dilation 和 return_indices 参数
class torch.nn.AvgPool2d(kernel_size, stride=None, padding=0, ceil_mode=False, count_include_pad=True, divisor_override=None)

# count_include_pad: 如果为True（默认），计算平均值时会将填充的0计入分母
# divisor_override: 如果指定，将用该值作为除数，而不是窗口内的元素个数

In [13]:
import torch
import torch.nn as nn

# 定义一个 4x4 输入
input_data = torch.tensor([[[[1.0, 2.0, 3.0, 4.0],
               [5.0, 6.0, 7.0, 8.0],
               [9.0, 10.0, 11.0, 12.0],
               [13.0, 14.0, 15.0, 16.0]]]])

# 1. 最大池化
max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
# 2. 平均池化
avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)

print("原始输入 (4x4):")
print(input_data[0,0])

print("\n最大池化结果 (取最大值):")
print(max_pool(input_data)[0,0])
print("\n平均池化结果 (取平均值):")
print(avg_pool(input_data)[0,0])

原始输入 (4x4):
tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.],
        [13., 14., 15., 16.]])

最大池化结果 (取最大值):
tensor([[ 6.,  8.],
        [14., 16.]])

平均池化结果 (取平均值):
tensor([[ 3.5000,  5.5000],
        [11.5000, 13.5000]])


---
##最大反池化MaxUnpoolNd
最大反池化根据前面最大池化得到的输出和相对应的索引，将最大值放置在对应的位置上，同时将其他位置补0.

最大反池化可以理解为固定规则的空间恢复，而转置卷积可以理解为可学习训练的上采样恢复大尺度特征。

In [ ]:
class torch.nn.MaxUnpool2d(kernel_size, stride=None, padding=0)

In [16]:
import torch
import torch.nn as nn

# 1. 模拟池化过程，获取索引
pool = nn.MaxPool2d(kernel_size=2, stride=2, return_indices=True)
unpool = nn.MaxUnpool2d(kernel_size=2, stride=2)

# 创建 4x4 输入
x = torch.tensor([[[[1.0, 2.0, 3.0, 4.0],
           [5.0, 6.0, 7.0, 8.0],
           [9.0, 10.0, 11.0, 12.0],
           [13.0, 14.0, 15.0, 16.0]]]])

# 池化：得到 2x2 的结果和索引
output, indices = pool(x)

# 2. 反池化过程：将 2x2 还原回 4x4
# 注意这里必须传入池化时记录的 indices
reconstructed_x = unpool(output, indices)

print("池化后的 2x2 结果:")
print(output[0,0])
print("\n反池化还原后的 4x4 矩阵 (最大值回到了原位，其余补0):")
print(reconstructed_x[0,0])

池化后的 2x2 结果:
tensor([[ 6.,  8.],
        [14., 16.]])

反池化还原后的 4x4 矩阵 (最大值回到了原位，其余补0):
tensor([[ 0.,  0.,  0.,  0.],
        [ 0.,  6.,  0.,  8.],
        [ 0.,  0.,  0.,  0.],
        [ 0., 14.,  0., 16.]])


---
##范数池化LPPoolNd
比较少见的池化方式，来源于范数的思想。
$$y
=
\left(
\sum_i |x_i|^p
\right)^{1/p}$$
p=1时近似为求和：
$$y=\sum_i |x_i|$$
p=2时相当于局部能量：
$$y=\sqrt{\sum_i x_i^2}$$
p→∞时，范数池化相当于最大池化

In [15]:
import torch.nn as nn
# LPPool2d：基于 Lp 范数的池化
class torch.nn.LPPool2d(norm_type, kernel_size, stride=None, ceil_mode=False)

# norm_type: Lp 范数中的 p。p=1 是求和池化，p=无穷大 是最大池化。
# kernel_size: 池化窗口大小
# stride: 步长，默认值等于 kernel_size

SyntaxError: invalid syntax (2868568368.py, line 3)

---
##分数最大池化FractionalMaxPoolNd
普通最大池化（ceiling模式），输出和输入的缩放比例是整数（由stride决定），分数最大池化则可以实现分数比例的缩放。具体地，分数最大池化通过随机数让滑动步长随机变化而不是固定为一个值，最终实现分数比例的缩放。

即，FractionalMaxPool = 用不均匀池化窗口位置实现非整数比例的 MaxPool 下采样。

In [ ]:
import torch.nn as nn
# 分数最大池化：支持非整数比例的下采样
class torch.nn.FractionalMaxPool2d(kernel_size, output_size=None, output_ratio=None, return_indices=False, _random_samples=None)

# kernel_size: 池化核尺寸
# output_size: 目标输出尺寸 (h, w)
# output_ratio: 输出与输入的比例 (例如 0.5)
# return_indices: 是否返回最大值索引

---
##自适应池化Adaptive Pooling
普通池化的逻辑是：给定kernel_size 和 stride，函数计算输出多大，或者说只是直接按参数进行池化，最后的输出得到多大就是多大。

自适应池化则相反：直接给定最后输出的尺寸大小，无论输入尺寸大小是多少，函数自动匹配所有的参数，得到要求尺寸大小的池化结果。




###AdaptiveMaxPoolNd自适应最大池化

In [ ]:
class torch.nn.AdaptiveMaxPool2d(output_size, return_indices=False)

###AdaptiveAvgPoolNd自适应平均池化

In [ ]:
class torch.nn.AdaptiveAvgPool2d(output_size)

自适应池化最大的作用就是处理不同尺寸的输入，控制输出尺寸相同，方便下一层的后续操作。

在实现方式上，系统自动根据要求的输出尺寸，将输入近似等分为对应个数的区域，在每个区域内进行池化操作，区域甚至可能有少量的重叠，核心目的就是保证最终输出尺寸相同。




##Pooling VS Stride Convollution
池化操作通常没有可训练参数，会影响网络计算和信息流，但一般不会增加模型参数量。考虑池化层的反向传播，MaxPool 的反向传播梯度只流向最大值对应的位置，AvgPool 的反向传播梯度平均分给窗口中的所有元素。

池化不是神经网络中必须存在的结构。普通 Pooling 所承担的下采样功能，很多时候可以用可学习的 stride convolution 实现。Stride convolution 自由度更高，可以学习如何在下采样过程中保留信息；而 Pooling 使用固定的 Max、Average 等规则，没有或几乎没有可学习参数，因此计算和训练成本更低，也减少了模型需要学习的自由度。当任务本身就适合保留局部最大响应或平均响应时，Pooling 的这种归纳偏置反而可能更加简单、稳定。Adaptive Pooling 则进一步允许直接指定输出空间尺寸，可以将不同尺寸的输入特征统一为固定尺寸，这是固定 stride convolution 不容易实现的。